# Amaliy mashgʻulot 8. Ketma-ketliklarni belgilash: yashirin Markov modellari (HMM) va Viterbi algoritmi

8-laboratoriyada Viterbi panjarasini qadamma-qadam toʻldirdingiz. Bu yerda xuddi shuni kodda bajaramiz:

1. kichik belgilangan korpus (UD UPOS teglari) boʻyicha oʻtish va emissiya ehtimollarini silliqlash bilan maksimal haqiqatga oʻxshashlik usulida baholaymiz;
2. Viterbi algoritmini noldan yozamiz va panjarani qadamma-qadam chop etamiz;
3. Viterbini toʻliq saralash bilan tekshiramiz va ochkoʻz (greedy) dekodlash bilan solishtiramiz;
4. ajratib qoʻyilgan gaplarda aniqlikni oʻlchaymiz va modelni notanish soʻzlarning tegini qoʻshimchasiga qarab topishga oʻrgatamiz (oʻzbek tili uchun bu ayniqsa muhim);
5. modelimizni `nltk` dagi HMM bilan solishtiramiz.

*«NLP algoritmlari» kursining 8-laboratoriyasiga amaliy mashgʻulot. Kataklarni yuqoridan pastga ishga tushiring: Shift+Enter yoki «Runtime → Run all». Javoblaringizni saqlash uchun «File → Save a copy in Drive» ni bosing.*

## 1. Korpus va HMM parametrlari

Ketma-ketliklarni belgilash (sequence labeling) — gapdagi har bir soʻzga belgi qoʻyish masalasi: soʻz turkumi, nomlangan obyekt turi va hokazo. Yashirin Markov modeli (HMM) teglar $t_1 \dots t_n$ Markov zanjirini tashkil qiladi, har bir soʻz $w_i$ esa oʻz tegi tomonidan «hosil qilinadi» deb hisoblaydi:
$$P(w, t) = \prod_{i=1}^{n} P(t_i \mid t_{i-1})\, P(w_i \mid t_i) \cdot P(\langle/s\rangle \mid t_n), \qquad t_0 = \langle s\rangle.$$

Ikkala jadvalni korpus boʻyicha baholaymiz. Quyida UD UPOS teglari bilan belgilangan 40 ga yaqin qisqa gap bor: NOUN — ot, VERB — feʼl, ADJ — sifat, ADV — ravish, PRON — olmosh, DET — aniqlovchi soʻz («bu», «shu», «har»), ADP — koʻmakchi («bilan», «uchun»), NUM — son, PART — yuklama, CCONJ — bogʻlovchi. Korpusga omonimlar ataylab kiritilgan: «yoz» (fasl / yozmoq), «ot» (hayvon / otmoq), «qoʻy» (hayvon / qoʻymoq), «och» (och qolmoq / ochmoq), «u» (olmosh / koʻrsatish soʻzi).

In [ ]:
TRAIN_TEXT = """
yoz/NOUN keldi/VERB
yoz/NOUN juda/ADV issiq/ADJ boʻldi/VERB
bu/DET yoz/NOUN biz/PRON qishloqqa/NOUN bordik/VERB
yoz/NOUN tez/ADV oʻtdi/VERB
ukam/NOUN xat/NOUN yozdi/VERB
ismingizni/NOUN shu/DET yerga/NOUN yozing/VERB
sen/PRON ham/PART bir/NUM sheʼr/NOUN yoz/VERB
ot/NOUN tez/ADV chopdi/VERB
bolalar/NOUN otga/NOUN pichan/NOUN berdi/VERB
toshni/NOUN suvga/NOUN ot/VERB
bu/DET ot/NOUN juda/ADV chiroyli/ADJ
qoʻy/NOUN dalada/NOUN oʻtlayapti/VERB
kitobni/NOUN stolga/NOUN qoʻy/VERB
choʻpon/NOUN qoʻylarni/NOUN togʻga/NOUN haydadi/VERB
uch/NUM bola/NOUN maktabga/NOUN bordi/VERB
men/PRON uch/NUM kitob/NOUN oldim/VERB
eshikni/NOUN och/VERB
mushuk/NOUN och/ADJ qoldi/VERB
talaba/NOUN kutubxonada/NOUN qiziqarli/ADJ kitob/NOUN oʻqiyapti/VERB
oʻqituvchi/NOUN oʻquvchilarga/NOUN yangi/ADJ kitoblar/NOUN berdi/VERB
biz/PRON katta/ADJ uyda/NOUN yashaymiz/VERB
u/PRON har/DET kuni/NOUN maktabga/NOUN boradi/VERB
onam/NOUN osh/NOUN pishirdi/VERB
akam/NOUN va/CCONJ singlim/NOUN bozorga/NOUN borishdi/VERB
bugun/ADV havo/NOUN juda/ADV sovuq/ADJ
men/PRON kecha/ADV yangi/ADJ kitob/NOUN sotib/VERB oldim/VERB
bolalar/NOUN hovlida/NOUN toʻp/NOUN oʻynashdi/VERB
ular/PRON doim/ADV kitob/NOUN oʻqishadi/VERB
bu/DET kitob/NOUN juda/ADV qiziqarli/ADJ
qushlar/NOUN osmonda/NOUN uchdi/VERB
otam/NOUN eski/ADJ derazani/NOUN tuzatdi/VERB
men/PRON doʻstim/NOUN bilan/ADP parkka/NOUN bordim/VERB
u/PRON men/PRON uchun/ADP gul/NOUN olib/VERB keldi/VERB
kecha/ADV kuchli/ADJ yomgʻir/NOUN yogʻdi/VERB
ikki/NUM doʻst/NOUN uzoq/ADV suhbatlashdi/VERB
besh/NUM oʻquvchi/NOUN masalani/NOUN yechdi/VERB
singlim/NOUN chiroyli/ADJ rasm/NOUN chizdi/VERB
talabalar/NOUN imtihonga/NOUN tayyorlanyapti/VERB
u/DET kitob/NOUN stolda/NOUN yotibdi/VERB
men/PRON kecha/ADV doʻstimga/NOUN xat/NOUN yozdim/VERB
oʻqituvchi/NOUN darsni/NOUN tushuntirdi/VERB
bu/DET yil/NOUN qish/NOUN erta/ADV keldi/VERB
"""
TEST_TEXT = """
bu/DET xatni/NOUN yoz/VERB
bolalar/NOUN bogʻda/NOUN oʻynadi/VERB
onam/NOUN mazali/ADJ non/NOUN yopdi/VERB
ot/NOUN dalada/NOUN chopyapti/VERB
men/PRON yangi/ADJ daftar/NOUN sotib/VERB oldim/VERB
ukam/NOUN kitobni/NOUN stolga/NOUN qoʻydi/VERB
talabalar/NOUN kutubxonada/NOUN jim/ADV oʻtirishdi/VERB
ikki/NUM qiz/NOUN chiroyli/ADJ qoʻshiq/NOUN aytdi/VERB
oʻqituvchi/NOUN yangi/ADJ mavzuni/NOUN tushuntirdi/VERB
qushlar/NOUN tez/ADV uchib/VERB ketdi/VERB
otam/NOUN shaharga/NOUN ketdi/VERB
ular/PRON menga/PRON kitob/NOUN berdi/VERB
"""

def read_tagged(text):
    # har bir qator — gap, har bir token soʻz/TEG koʻrinishida yozilgan
    return [[tuple(tok.rsplit("/", 1)) for tok in line.split()]
            for line in text.strip().splitlines() if line.strip()]

train = read_tagged(TRAIN_TEXT)
test = read_tagged(TEST_TEXT)
train_vocab = {w for s in train for w, _ in s}
print(f"oʻqitish: {len(train)} gap, {sum(map(len, train))} token")
print(f"test:    {len(test)} gap, {sum(map(len, test))} token")
print(f"testdagi notanish soʻzlar: {sum(w not in train_vocab for s in test for w, _ in s)}")
print(train[0])

Maksimal haqiqatga oʻxshashlik bahosi (MLE) — oddiy ulushlar: $P(t_i \mid t_{i-1}) = \frac{C(t_{i-1}, t_i)}{C(t_{i-1})}$ va $P(w \mid t) = \frac{C(t, w)}{C(t)}$. Kichik korpusda koʻp juftliklar bir marta ham uchramaydi. MLE ularga nol beradi, bitta nol esa butun koʻpaytmani nolga aylantiradi. Shuning uchun har bir sanoqqa kichik $k$ qoʻshamiz (add-k silliqlash, 4-amaliy mashgʻulotga qarang): oʻtishlar uchun $k = 0{,}1$, emissiyalar uchun $k = 0{,}01$. Emissiyalar lugʻatida notanish soʻz uchun bitta joy ajratamiz.

Omonimlarni koʻramiz: har bir soʻz har bir teg bilan necha marta uchragan va qanday $\log P(w \mid t)$ chiqqan.

In [ ]:
import numpy as np
import pandas as pd
from collections import Counter
pd.set_option("display.width", 200)

# (oldingi teg, teg) juftlari, (teg, soʻz) juftlari va teglar chastotasini sanaymiz
START, END = "<s>", "</s>"
trans_count, emit_count, tag_count = Counter(), Counter(), Counter()
for sent in train:
    prev = START
    for w, t in sent:
        trans_count[prev, t] += 1
        emit_count[t, w] += 1
        tag_count[t] += 1
        prev = t
    trans_count[prev, END] += 1

TAGS = sorted(tag_count)
VOCAB = sorted(train_vocab)
prev_states = [START] + TAGS     # qatorlar: qayerdan
next_states = TAGS + [END]       # ustunlar: qayerga

# add-k silliqlash: oʻtishlar va emissiyalar
K_TRANS, K_EMIT = 0.1, 0.01
C = np.array([[trans_count[p, t] for t in next_states] for p in prev_states], dtype=float)
A = (C + K_TRANS) / (C.sum(axis=1, keepdims=True) + K_TRANS * len(next_states))
logA = np.log(A)
tag_total = np.array([tag_count[t] for t in TAGS], dtype=float)

def emit_logp(word):
    # notanish soʻz uchun sanoq 0, faqat silliqlash qoladi; lugʻatdagi +1 — notanish soʻz uchun joy
    counts = np.array([emit_count[t, word] for t in TAGS], dtype=float)
    return np.log((counts + K_EMIT) / (tag_total + K_EMIT * (len(VOCAB) + 1)))

print("Oʻtishlar sanogʻi C(oldingi → keyingi):")
print(pd.DataFrame(C.astype(int), index=prev_states, columns=next_states).to_string())
print("\nOʻtish ehtimollari P(keyingi | oldingi), silliqlangan:")
print(pd.DataFrame(A, index=prev_states, columns=next_states).round(2).to_string())
print()

print("Omonimlar: C(teg, soʻz) sanoqlari va log P(soʻz | teg)")
for w in ["yoz", "ot", "qoʻy", "och", "u"]:
    parts = [f"{t} {emit_count[t, w]} ({emit_logp(w)[j]:.2f})" for j, t in enumerate(TAGS) if emit_count[t, w]]
    print(f"   {w:<8}", " | ".join(parts))

Oʻtishlar jadvali rasmda. Oʻzbek tilida feʼl gap oxirida turadi (SOV), shuning uchun VERB → </s> ehtimoli juda katta, NOUN → </s> esa deyarli nol. Bu ustun Viterbi uchun juda foydali boʻladi.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(A, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(len(next_states)), next_states, rotation=45)
ax.set_yticks(range(len(prev_states)), prev_states)
for i in range(A.shape[0]):
    for j in range(A.shape[1]):
        if A[i, j] >= 0.05:
            ax.text(j, i, f"{A[i, j]:.2f}", ha="center", va="center", fontsize=8,
                    color="white" if A[i, j] > 0.5 else "black")
ax.set_xlabel("keyingi teg")
ax.set_ylabel("oldingi teg")
ax.set_title("Oʻtish ehtimollari P(teg | oldingi teg)")
plt.colorbar(im, fraction=0.04)
plt.tight_layout()
plt.show()

## 2. Viterbi algoritmi: panjara qadamma-qadam

Teglarning barcha ketma-ketliklarini saralab chiqish mumkin emas: ular $|T|^n$ ta. Viterbi algoritmi masalani dinamik dasturlash bilan yechadi. $\delta_i(t)$ — $i$-soʻzda $t$ tegi bilan tugaydigan eng yaxshi yoʻl ehtimolining logarifmi:
$$\delta_i(t) = \max_{t'} \big[\delta_{i-1}(t') + \log P(t \mid t')\big] + \log P(w_i \mid t).$$
$\psi_i(t)$ koʻrsatkichi maksimum qaysi $t'$ dan kelganini eslab qoladi. Oxirida $\langle/s\rangle$ ga oʻtishni qoʻshamiz va koʻrsatkichlar boʻyicha orqaga yuramiz. Logarifmlarda hisoblaymiz, chunki koʻp kichik ehtimollar koʻpaytmasi tezda mashina noliga aylanadi.

Misol: «sen xat yoz». Korpusda «yoz» koʻproq ot (fasl), lekin bu yerda feʼl (buyruq mayli). Laboratoriyadagi misollarni («yoz keldi», «ot chopdi») 3-boʻlimda koʻramiz.

In [ ]:
def viterbi(words, emit=emit_logp, trans=None, verbose=False):
    L = logA if trans is None else trans
    n, T = len(words), len(TAGS)
    E = np.array([emit(w) for w in words])          # E[i, j] = log P(soʻz_i | teg_j) — barcha soʻzlar uchun oldindan
    delta = np.zeros((n, T))
    back = np.zeros((n, T), dtype=int)
    delta[0] = L[0, :T] + E[0]
    for i in range(1, n):
        scores = delta[i - 1][:, None] + L[1:, :T]      # scores[k, j] = δ[i-1, k] + log P(teg_j | teg_k): j tegiga qaysi tegdan kelish foydaliroq
        back[i] = scores.argmax(axis=0)
        delta[i] = scores.max(axis=0) + E[i]
    final = delta[-1] + L[1:, T]                        # gap oxiri </s> ga oʻtish
    path = [int(final.argmax())]
    for i in range(n - 1, 0, -1):                       # koʻrsatkichlar boʻyicha orqaga yurish
        path.append(int(back[i, path[-1]]))
    path.reverse()
    if verbose:
        show_trellis(words, E, delta, back, final, path)
    return [TAGS[j] for j in path], float(final.max()), delta, back

def show_trellis(words, E, delta, back, final, path):
    for i, w in enumerate(words):
        print(f"soʻz {i + 1}: «{w}»  (eng yaxshi 3 ta teg koʻrsatilgan, qolganlari yomonroq)")
        for j in np.argsort(-delta[i])[:3]:
            t = TAGS[j]
            if i == 0:
                print(f"   δ({t:<5}) = log P({t}|<s>) {logA[0, j]:6.2f} + log P({w}|{t}) {E[i, j]:6.2f} = {delta[i, j]:7.2f}")
            else:
                k = back[i, j]
                print(f"   δ({t:<5}) = δ({TAGS[k]}) {delta[i - 1, k]:6.2f} + log P({t}|{TAGS[k]}) {logA[k + 1, j]:6.2f}"
                      f" + log P({w}|{t}) {E[i, j]:6.2f} = {delta[i, j]:7.2f}   ← qayerdan {TAGS[k]}")
    print("oxiri:")
    for j in np.argsort(-final)[:3]:
        print(f"   δ({TAGS[j]:<5}) {delta[-1, j]:6.2f} + log P(</s>|{TAGS[j]}) {logA[j + 1, len(TAGS)]:6.2f} = {final[j]:7.2f}")
    print("eng yaxshi yoʻl:", " ".join(f"{w}/{TAGS[j]}" for w, j in zip(words, path)))

demo = "sen xat yoz".split()
best_tags, best_score, delta, back = viterbi(demo, verbose=True)
print("log P(soʻzlar, teglar) =", round(best_score, 2))

Xuddi shu panjara rasmda. Ramkalar va strelkalar topilgan yoʻlni koʻrsatadi. «yoz» ustuniga qarang: NOUN va VERB yaqin, hal qiluvchi omil — gap oxiriga oʻtish: VERB → </s> ehtimoli NOUN → </s> dan yuzlab marta katta.

In [ ]:
def plot_trellis(words, delta, tags):
    rel = (delta - delta.max(axis=1, keepdims=True)).T   # rang: katak oʻz ustunidagi eng yaxshisidan qanchalik yomon (0 = eng yaxshi)
    fig, ax = plt.subplots(figsize=(1.7 * len(words) + 2, 5))
    ax.imshow(np.maximum(rel, -12), cmap="Blues", vmin=-12, vmax=0, aspect="auto")
    for i in range(len(words)):
        for j in range(len(TAGS)):
            ax.text(i, j, f"{delta[i, j]:.1f}", ha="center", va="center", fontsize=8,
                    color="white" if rel[j, i] > -3 else "black")
    path = [TAGS.index(t) for t in tags]
    for i in range(1, len(words)):
        ax.annotate("", xy=(i - 0.3, path[i]), xytext=(i - 0.7, path[i - 1]),
                    arrowprops=dict(arrowstyle="->", color="crimson", lw=2))
    for i, j in enumerate(path):
        ax.add_patch(plt.Rectangle((i - 0.5, j - 0.5), 1, 1, fill=False, ec="crimson", lw=2))
    ax.set_xticks(range(len(words)), words)
    ax.set_yticks(range(len(TAGS)), TAGS)
    ax.set_title("Viterbi panjarasi: son — δ, rang — ustundagi eng yaxshi katakdan orqada qolish", fontsize=10)
    plt.tight_layout()
    plt.show()

plot_trellis(demo, delta, best_tags)

## 3. Tekshiruv: toʻliq saralash va ochkoʻz algoritm

Toʻliq saralash barcha $|T|^n$ yoʻl uchun $\log P(w, t)$ ni hisoblaydi. 2–3 soʻz va 10 teg uchun bu 100–1 000 yoʻl: ularni hali saralash mumkin, javob esa Viterbi bilan albatta mos kelishi kerak. Ochkoʻz algoritm chapdan oʻngga yuradi, joriy soʻz uchun oldingi tegni hisobga olib eng yaxshi tegni tanlaydi va qarorini boshqa oʻzgartirmaydi. U keyingi soʻzlarni va gap oxirini koʻrmaydi. Feʼl oxirda turadigan oʻzbek tilida bu ayniqsa xavfli.

In [ ]:
import itertools

def seq_logp(words, tags, emit=emit_logp):
    idx = [TAGS.index(t) for t in tags]
    lp = logA[0, idx[0]] + emit(words[0])[idx[0]]
    for i in range(1, len(words)):
        lp += logA[idx[i - 1] + 1, idx[i]] + emit(words[i])[idx[i]]
    return lp + logA[idx[-1] + 1, len(TAGS)]

def brute_force(words):
    # toʻliq saralash: |T|^n ta yoʻl
    E = [emit_logp(w) for w in words]
    best, best_lp, n_paths = None, -np.inf, 0
    for idx in itertools.product(range(len(TAGS)), repeat=len(words)):
        lp = logA[0, idx[0]] + E[0][idx[0]]
        for i in range(1, len(words)):
            lp += logA[idx[i - 1] + 1, idx[i]] + E[i][idx[i]]
        lp += logA[idx[-1] + 1, len(TAGS)]
        n_paths += 1
        if lp > best_lp:
            best, best_lp = [TAGS[j] for j in idx], lp
    return best, best_lp, n_paths

def greedy(words, emit=emit_logp):
    # ochkoʻz: joriy soʻz uchun eng yaxshi teg, qaror boshqa oʻzgarmaydi
    prev, out = 0, []
    for w in words:
        j = int((logA[prev, :len(TAGS)] + emit(w)).argmax())
        out.append(TAGS[j])
        prev = j + 1
    return out

for s in ["sen xat yoz", "yoz keldi", "ot chopdi", "toshni suvga ot", "mushuk och qoldi"]:
    words = s.split()
    v_tags, v_lp, _, _ = viterbi(words)
    b_tags, b_lp, n_paths = brute_force(words)
    g_tags = greedy(words)
    print(f"gap: {s}")
    print(f"   {'Viterbi':<9}: {' '.join(v_tags):<28} log P = {v_lp:7.2f}")
    print(f"   {'saralash':<9}: {' '.join(b_tags):<28} log P = {b_lp:7.2f}  ({n_paths} ta yoʻl saralandi)")
    g_lp = seq_logp(words, g_tags)
    mark = "Viterbi bilan bir xil" if g_tags == v_tags else "FARQ QILADI"
    print(f"   {'ochkoʻz':<9}: {' '.join(g_tags):<28} log P = {g_lp:7.2f}  {mark}\n")

Nega toʻliq saralash haqiqiy gaplar uchun yaramaydi: Viterbi $n \cdot |T|^2$ amal bajaradi, saralash esa $|T|^n$. 15 soʻzli gap va 10 teg uchun bu 1 500 ga qarshi $10^{15}$.

In [ ]:
n = np.arange(1, 16)
T = len(TAGS)
plt.figure(figsize=(6, 3.5))
plt.semilogy(n, n * T ** 2, "o-", label="Viterbi: n·|T|²")
plt.semilogy(n, float(T) ** n, "s-", label="saralash: |T|ⁿ")
plt.xlabel("gap uzunligi n")
plt.ylabel("amallar (log. shkala)")
plt.title("Amallar soni: Viterbi va toʻliq saralash")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Yangi gaplarda aniqlik va notanish soʻzlar

Modelni 12 ta ajratib qoʻyilgan gapda tekshiramiz. Oʻqitishda boʻlmagan soʻz (OOV, out of vocabulary) $P(w \mid t) = \frac{k}{C(t) + k(|V|+1)}$ ni oladi. Bu qiymat kam uchraydigan teglarda kattaroq (maxraj kichik), shuning uchun notanish soʻzlar koʻpincha gʻalati teglar oladi. Agglyutinativ tilda bu muammo kuchliroq: bitta oʻzakning oʻnlab shakli bor («xat», «xatni», «xatlar», «xatimni»…), kichik korpusda ularning aksariyati uchramaydi.

In [ ]:
def evaluate(data, emit=emit_logp, trans=None, show_errors=False):
    # tanish va notanish (OOV) soʻzlar uchun aniqlik alohida
    ok = {"known": 0, "unk": 0}
    total = {"known": 0, "unk": 0}
    errors = []
    for sent in data:
        words = [w for w, _ in sent]
        gold = [t for _, t in sent]
        pred, _, _, _ = viterbi(words, emit=emit, trans=trans)
        for w, g, p in zip(words, gold, pred):
            kind = "known" if w in train_vocab else "unk"
            total[kind] += 1
            ok[kind] += g == p
        if pred != gold:
            errors.append(" ".join(
                f"{w}{'' if w in train_vocab else '*'}/{p}" + ("" if p == g else f"({g})")
                for w, g, p in zip(words, gold, pred)))
    acc = {k: ok[k] / total[k] for k in ok}
    acc["all"] = (ok["known"] + ok["unk"]) / (total["known"] + total["unk"])
    print(f"tanish: {ok['known']}/{total['known']} = {acc['known']:.0%}   "
          f"notanish: {ok['unk']}/{total['unk']} = {acc['unk']:.0%}   jami: {acc['all']:.0%}")
    if show_errors and errors:
        print("Xatolar (* — notanish soʻz, qavs ichida — toʻgʻri teg):")
        for e in errors:
            print("   ", e)
    return acc

acc_plain = evaluate(test, show_errors=True)

Buni TnT teggeri gʻoyasi bilan tuzatamiz: oʻzbek tilida soʻz turkumini qoʻshimcha yaxshi koʻrsatadi: «-di», «-dim», «-yapti» feʼllarda, «-ni», «-ga», «-lar» otlarda. Yordamchi soʻzlar (koʻmakchilar, bogʻlovchilar, olmoshlar) yopiq turkumlar, ularning deyarli hammasi lugʻatda bor, shuning uchun notanish soʻzga faqat ochiq teglarni ruxsat beramiz: NOUN, VERB, ADJ, ADV. Shu turkumlardagi oʻqitish soʻzlari boʻyicha uzunligi 1–3 boʻlgan oxirgi harflar uchun $P(t \mid \text{qoʻshimcha})$ ni hisoblaymiz, kamida ikki marta uchragan eng uzun qoʻshimchani olamiz, teglarning umumiy chastotasiga qarab silliqlaymiz va Bayes formulasi bilan quyidagini olamiz:
$$P(w \mid t) = \frac{P(t \mid w)\,P(w)}{P(t)} \propto \frac{P(t \mid \text{qoʻshimcha})}{P(t)}.$$
$P(w)$ koʻpaytuvchisi barcha teglar uchun bir xil va yoʻl tanloviga taʼsir qilmaydi. Bu haqiqiy morfologik tahlil emas, faqat oxirgi harflar statistikasi: masalan, «-da» qoʻshimchasi oʻrin-payt kelishigi, lekin bizning korpusda «juda» soʻzi ham shu harflar bilan tugaydi.

In [ ]:
from collections import defaultdict

OPEN_TAGS = ["ADJ", "ADV", "NOUN", "VERB"]   # notanish soʻz deyarli har doim ochiq turkumdan: yordamchi soʻzlarning deyarli hammasi lugʻatda bor
is_open = np.array([t in OPEN_TAGS for t in TAGS])
SUFFIX_MAX, MIN_COUNT = 3, 2
suffix_count = defaultdict(Counter)       # uzunligi 1..SUFFIX_MAX boʻlgan oxirgi harflar uchun teglar sanogʻi
for sent in train:
    for w, t in sent:
        if t in OPEN_TAGS:
            for k in range(1, SUFFIX_MAX + 1):
                if len(w) > k:
                    suffix_count[w[-k:]][t] += 1
tag_prior = np.where(is_open, tag_total, 0) / tag_total[is_open].sum()

def suffix_tag_probs(word, m=1.0):
    # kamida MIN_COUNT marta uchragan eng uzun qoʻshimcha; teglarning umumiy chastotasiga qarab silliqlaymiz
    for k in range(SUFFIX_MAX, 0, -1):
        c = suffix_count.get(word[-k:])
        if c and sum(c.values()) >= MIN_COUNT:
            counts = np.array([c[t] for t in TAGS], dtype=float)
            return (counts + m * tag_prior) / (counts.sum() + m), word[-k:]
    return tag_prior, ""

def emit_logp_suffix(word):
    if word in train_vocab:
        return emit_logp(word)
    p, _ = suffix_tag_probs(word)
    # Bayes: P(w|t) ∝ P(t|qoʻshimcha) / P(t); barcha teglar uchun umumiy koʻpaytuvchi yoʻlga taʼsir qilmaydi
    out = np.full(len(TAGS), -np.inf)
    out[is_open] = np.log(p[is_open] / tag_prior[is_open]) + np.log(1 / len(VOCAB))
    return out

print("Oʻqitish soʻzlari boʻyicha P(teg | qoʻshimcha) (eng koʻp uchragan 3 ta teg):")
for suf in ["di", "ni", "ga", "da", "lar", "dim"]:
    c = suffix_count[suf]
    top = ", ".join(f"{t} {x / sum(c.values()):.2f}" for t, x in c.most_common(3))
    print(f"   -{suf:<4} n={sum(c.values()):<3} {top}")

Ikki modelni solishtiramiz. Tanish soʻzlar bir xil belgilanadi, butun farq notanish soʻzlarda. Oxirida koʻrinadiki, qoʻshimcha faqat ishora: «bogʻda» ning «-da» si korpusda «juda» tufayli ravishga ham oʻxshaydi, lekin otdan keyin va feʼldan oldin Viterbi NOUN ni tanlaydi. Emissiya va oʻtishlar birgalikda hal qiladi. «menga» xatosi agglyutinativ tilga xos: olmosh ham kelishik qoʻshimchasini oladi (men + -ga), shuning uchun yopiq turkum soʻzi lugʻatda yoʻq, «-ga» esa otni koʻrsatadi. Rus tilida «мне» kabi shakllar ham bor, lekin oʻzbek tilida bunday shakllar qoidaga muvofiq yasaladi va juda koʻp.

In [ ]:
print("qoʻshimchasiz HMM:")
acc_plain = evaluate(test)
print("HMM + qoʻshimchalar:")
acc_suffix = evaluate(test, emit=emit_logp_suffix, show_errors=True)

print("\nTestdagi notanish soʻzlar: faqat qoʻshimcha boʻyicha teg va Viterbi qoʻshnilarni hisobga olib tanlagan teg:")
for sent in test:
    words = [w for w, _ in sent]
    pred, _, _, _ = viterbi(words, emit=emit_logp_suffix)
    for w, (_, gold), p_ctx in zip(words, sent, pred):
        if w not in train_vocab:
            p, suf = suffix_tag_probs(w)
            alone = TAGS[int(np.argmax(np.where(is_open, p / np.where(is_open, tag_prior, 1), -1)))]
            mark = "✓" if p_ctx == gold else "✗ " + gold
            print(f"   {w:<14} -{suf:<4} qoʻshimcha: {alone:<5} Viterbi: {p_ctx:<5} {mark}")

labels = ["tanish", "notanish", "jami"]
x = np.arange(3)
plt.figure(figsize=(6, 3.2))
plt.bar(x - 0.2, [acc_plain[k] for k in ("known", "unk", "all")], 0.4, label="qoʻshimchasiz HMM", color="#9AA9C9")
plt.bar(x + 0.2, [acc_suffix[k] for k in ("known", "unk", "all")], 0.4, label="HMM + qoʻshimchalar", color="#2B55C9")
plt.xticks(x, labels)
plt.ylim(0, 1.05)
plt.ylabel("toʻgʻri teglar ulushi")
plt.title("Testdagi aniqlik")
plt.legend(loc="lower left")
plt.tight_layout()
plt.show()

## 5. Xuddi shu kutubxona bilan: `nltk`

`nltk` da tayyor HMM bor: `HiddenMarkovModelTrainer.train_supervised` xuddi shu chastotalarni sanaydi va silliqlaydi (`LidstoneProbDist` — oʻsha add-k silliqlash). Ikki farq bor: `nltk` modelida barcha jadvallar uchun bitta $k$ va gap oxiri $\langle/s\rangle$ ga oʻtish yoʻq. Bizning HMM ni xuddi shunday sozlasak, belgilash `nltk` bilan toʻliq mos kelishi kerak — bu kodimizni yaxshi tekshiradi. Keyin aniqlikni solishtiramiz.

In [ ]:
from nltk.tag.hmm import HiddenMarkovModelTrainer
from nltk.probability import LidstoneProbDist

# nltk (soʻz, teg) juftlaridan iborat gaplar roʻyxatini kutadi — bizning train kabi
trainer = HiddenMarkovModelTrainer(states=list(TAGS), symbols=list(VOCAB))
nltk_hmm = trainer.train_supervised(train, estimator=lambda fd, bins: LidstoneProbDist(fd, K_EMIT, bins))

# bizning HMM «nltk rejimida»: barcha jadvallar uchun bitta k va </s> ga oʻtishsiz
T = len(TAGS)
logA_nltk = np.zeros_like(logA)      # oxirgi ustun (</s>) nol boʻlib qoladi: u hech narsaga taʼsir qilmaydi
logA_nltk[:, :T] = np.log((C[:, :T] + K_EMIT) / (C[:, :T].sum(axis=1, keepdims=True) + K_EMIT * T))

def emit_logp_nltk(word):
    counts = np.array([emit_count[t, word] for t in TAGS], dtype=float)
    return np.log((counts + K_EMIT) / (tag_total + K_EMIT * len(VOCAB)))

sents = [[w for w, _ in s] for s in test] + [s.split() for s in ["sen xat yoz", "yoz keldi", "ot chopdi", "toshni suvga ot", "mushuk och qoldi"]]
same = sum(viterbi(ws, emit=emit_logp_nltk, trans=logA_nltk)[0] == [t for _, t in nltk_hmm.tag(ws)] for ws in sents)
print(f"bizning HMM nltk rejimida va nltk bir xil teglar bergan gaplar: {same}/{len(sents)}\n")

ok = sum(p == g for s in test for p, (_, g) in zip([t for _, t in nltk_hmm.tag([w for w, _ in s])], s))
print(f"nltk HMM            : {ok}/{sum(map(len, test))} = {ok / sum(map(len, test)):.0%}")
print("bizniki, nltk rejimi:", end=" ")
evaluate(test, emit=emit_logp_nltk, trans=logA_nltk)
print("bizniki, </s> bilan :", end=" ")
evaluate(test)

print("\n3-boʻlimdagi misollar, ularda toʻliq model va nltk farq qiladi:")
for s in ["sen xat yoz", "yoz keldi", "ot chopdi", "toshni suvga ot", "mushuk och qoldi"]:
    words = s.split()
    ours = viterbi(words)[0]
    theirs = [t for _, t in nltk_hmm.tag(words)]
    if ours != theirs:
        print(f"   {s:<26} bizniki: {' '.join(ours):<22} nltk: {' '.join(theirs)}")

«nltk rejimida» bizning kod aynan oʻsha teglarni beradi, demak u toʻgʻri. Toʻliq modelning aniqligi esa ancha yuqori. Sababi — $\langle/s\rangle$ ga oʻtish: `nltk` modeli oʻzbek gapi feʼl bilan tugashini bilmaydi, shuning uchun «sen xat yoz» dagi «yoz» ni ot deb belgilaydi. SOV tillar uchun gap oxiri modeli juda muhim: SVO tillar (ingliz, rus) uchun yaratilgan vositalar bu narsani eʼtiborsiz qoldirishi mumkin.

## 6. Topshiriqlar

1. **Silliqlash.** `K_EMIT` = 0.001, 0.1, 1 va `K_TRANS` = 0.01, 1 qiymatlarini sinab koʻring (oʻzgartirgandan keyin 1–4-boʻlim kataklarini qayta ishga tushiring). Tanish va notanish soʻzlardagi aniqlik qanday oʻzgaradi? Nega katta `K_EMIT` omonimlarga zarar qiladi?
2. **Toʻgʻri yoʻl algoritmi (forward).** `viterbi` ni nusxalang va maksimumni logarifmlardagi yigʻindiga almashtiring (`np.logaddexp.reduce(..., axis=0)`). Natijada $\log P(w)$ — barcha yoʻllar boʻyicha yigʻilgan gap ehtimoli chiqadi. Javobni saralash bilan tekshiring: `np.logaddexp.reduce([seq_logp(words, t) for t in itertools.product(TAGS, repeat=len(words))])`.
3. **Ochkoʻz algoritm qayerda xato qiladi.** Tanish soʻzlardan yana ikkita gap tuzing (masalan, «ot», «qoʻy», «och» soʻzlari bilan), ularda ochkoʻz algoritm va Viterbi farq qilsin. `viterbi(..., verbose=True)` panjarasini chop eting va qaysi oʻtish hal qilganini tushuntiring.
4. **Qoʻshimchalar qoidalari.** Oʻzbek feʼl qoʻshimchalari uchun qoida qoʻshing: `emit_logp_suffix` da notanish soʻz «-di», «-dim», «-yapti», «-moqda», «-gan» bilan tugasa, VERB ehtimolini oshiring. Notanish soʻzlardagi aniqlik oʻzgaradimi? `SUFFIX_MAX` (1, 2, 4) ni ham oʻzgartirib koʻring.
5. **Xatolar matritsasi.** Qoʻshimchali model uchun test gaplarida «toʻgʻri teg × bashorat qilingan teg» jadvalini tuzing (`pd.crosstab`). Qaysi teg juftlari koʻproq adashtiriladi?

*Maslahat: oʻzingiz 20–30 ta gapni belgilab, `my_corpus.txt` faylini «Files» paneli orqali yuklashingiz va uning matnini `TRAIN_TEXT` oʻrniga qoʻyishingiz mumkin.*

In [ ]:
# Kodingizni shu yerga yozing

In [ ]:
# Kodingizni shu yerga yozing

In [ ]:
# Kodingizni shu yerga yozing

In [ ]:
# Kodingizni shu yerga yozing

In [ ]:
# Kodingizni shu yerga yozing